# LipidOS — chunk + paper embedding on A100

Re-embeds the corpus with `scripts/embed.py` unchanged (BGE for prose chunks,
SPECTER2 for paper-level similarity). 16k chunks = 27 min on the local M2 MPS;
this corpus is now 12,563 chunks (4,632 new `source='pdf'` rows added by
`parse_pdf.py`, currently `vector_row_idx=NULL` and invisible to retrieval).
An A100 makes the re-embed loop cheap enough to redo whenever the corpus
definition changes.

**Runtime:** set **Runtime → Change runtime type → A100 GPU** before running.

**Flow:** upload `papers.db` → embed on the A100 → download the updated
`papers.db` + `chunk_vectors.npy` + `paper_vectors.npy`, replace the local
copies in `data/`.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > A100 GPU.'
print('CUDA device:', torch.cuda.get_device_name(0))

## 2. Install pinned dependencies

Same pins as `requirements.txt` — `adapters` silently upgrades `transformers`
underneath you if installed first; keep the order and pins as-is.

In [ ]:
!pip install -q transformers==4.57.6 protobuf==5.29.3 adapters==1.3.0
print('installed')

## 3. Upload `papers.db`

Locally:
```bash
cd ~/resume_projects/LipidOS && cp data/papers.db papers.db
```
Then run this cell and pick that `papers.db`.

In [ ]:
import os
from google.colab import files

os.makedirs('/content/data', exist_ok=True)
up = files.upload()
fname = next(iter(up))
os.rename(fname, '/content/data/papers.db')
print('papers.db ready at /content/data/papers.db')

## 4. Write `embed.py` (unchanged from `scripts/embed.py`)

Reused verbatim rather than re-derived here, so the embedding logic run on the
A100 can never silently drift from what's in the repo.

In [ ]:
%%writefile /content/embed.py
"""Embed parsed content into flat numpy vector files.

  chunks  -> BGE (bge-base-en-v1.5)  -> chunk_vectors.npy  (n_chunks, 768)
  papers  -> SPECTER2                -> paper_vectors.npy  (n_papers, 768)

Why two models: they do different jobs. BGE is trained on query->passage pairs,
which is literally chunk retrieval. SPECTER2 is trained on citation graphs to
place whole *papers* near each other from title+abstract -- document similarity,
not passage search. Using SPECTER2 for chunks would be a task mismatch; using
BGE for paper-level similarity would throw away the citation-graph signal.

SQLite holds everything relational; the .npy files hold only floats, because
cosine similarity is a numpy operation and SQLite has no fast vector search.
`vector_row_idx` is the join key: chunks.vector_row_idx == row i of the array.

Usage:
    python scripts/embed.py                # both tracks
    python scripts/embed.py --only chunks
"""

import argparse
import sqlite3
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

ROOT = Path(__file__).resolve().parent.parent
DATA = ROOT / "data"
DB_PATH = DATA / "papers.db"

BGE_MODEL = "BAAI/bge-base-en-v1.5"
SPECTER_MODEL = "allenai/specter2_base"
SPECTER_ADAPTER = "allenai/specter2"   # proximity adapter -- see encode()
MAX_LEN = 512


def device() -> str:
    if torch.backends.mps.is_available():
        return "mps"   # Apple Silicon GPU
    return "cuda" if torch.cuda.is_available() else "cpu"


def encode(texts: list[str], model_name: str, dev: str, batch_size: int = 16,
           adapter: str | None = None) -> np.ndarray:
    """CLS-pool + L2-normalise. Both BGE and SPECTER2 are BERT-family CLS models.

    `adapter` loads a SPECTER2 task adapter. The bare specter2_base encoder is
    undertrained for similarity: without the proximity adapter every paper in a
    topically-narrow corpus scores ~0.92 against every other, so the ranking is
    within noise. The adapter is what SPECTER2 actually is.
    """
    tok = AutoTokenizer.from_pretrained(model_name)
    if adapter:
        from adapters import AutoAdapterModel
        model = AutoAdapterModel.from_pretrained(model_name)
        model.load_adapter(adapter, source="hf", load_as="proximity",
                           set_active=True)
        model = model.to(dev).eval()
    else:
        model = AutoModel.from_pretrained(model_name).to(dev).eval()

    out = []
    with torch.inference_mode():
        for i in range(0, len(texts), batch_size):
            batch = tok(texts[i:i + batch_size], padding=True, truncation=True,
                        max_length=MAX_LEN, return_tensors="pt").to(dev)
            hidden = model(**batch).last_hidden_state[:, 0]  # CLS
            out.append(F.normalize(hidden, p=2, dim=1).cpu().numpy())
            done = min(i + batch_size, len(texts))
            print(f"\r  {done}/{len(texts)}", end="", flush=True)
    print()
    return np.vstack(out).astype(np.float32)


def embed_chunks(db: sqlite3.Connection, dev: str) -> None:
    rows = db.execute(
        """SELECT c.chunk_id, c.section_path, c.text, p.title
           FROM chunks c JOIN papers p USING(paper_id) ORDER BY c.chunk_id"""
    ).fetchall()
    if not rows:
        print("no chunks -- run parse_jats.py first", file=sys.stderr)
        return

    # Prepend title + section path. A bare paragraph loses its context: "the band
    # shifts to 1655" is unretrievable without knowing which paper and which
    # section it came from.
    texts = [f"{r[3]}\n{r[1]}\n\n{r[2]}" if r[1] else f"{r[3]}\n\n{r[2]}"
             for r in rows]

    print(f"embedding {len(texts)} chunks with {BGE_MODEL} on {dev}")
    vecs = encode(texts, BGE_MODEL, dev)
    np.save(DATA / "chunk_vectors.npy", vecs)

    db.executemany("UPDATE chunks SET vector_row_idx=? WHERE chunk_id=?",
                   [(i, r[0]) for i, r in enumerate(rows)])
    db.commit()
    print(f"  -> chunk_vectors.npy {vecs.shape}")


def embed_papers(db: sqlite3.Connection, dev: str) -> None:
    rows = db.execute(
        "SELECT paper_id, title, abstract FROM papers ORDER BY paper_id").fetchall()
    if not rows:
        return

    # SPECTER2's training format: title [SEP] abstract.
    sep = AutoTokenizer.from_pretrained(SPECTER_MODEL).sep_token
    texts = [f"{r[1] or ''}{sep}{r[2] or ''}" for r in rows]

    print(f"embedding {len(texts)} papers with {SPECTER_MODEL} + proximity adapter on {dev}")
    vecs = encode(texts, SPECTER_MODEL, dev, adapter=SPECTER_ADAPTER)
    np.save(DATA / "paper_vectors.npy", vecs)

    db.executemany("INSERT OR REPLACE INTO paper_embeddings VALUES (?,?)",
                   [(r[0], i) for i, r in enumerate(rows)])
    db.commit()
    print(f"  -> paper_vectors.npy {vecs.shape}")


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--only", choices=["chunks", "papers"], default=None)
    args = ap.parse_args()

    dev = device()
    db = sqlite3.connect(DB_PATH)
    if args.only != "papers":
        embed_chunks(db, dev)
    if args.only != "chunks":
        embed_papers(db, dev)
    db.close()
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## 5. Run both tracks

In [ ]:
import time
t0 = time.time()
!cd /content && python embed.py
print(f'\nembed.py finished in {(time.time()-t0)/60:.1f} min')

## 6. Sanity check shapes

In [ ]:
import numpy as np, sqlite3
chunk_vecs = np.load('/content/data/chunk_vectors.npy')
paper_vecs = np.load('/content/data/paper_vectors.npy')
db = sqlite3.connect('/content/data/papers.db')
n_chunks = db.execute('SELECT COUNT(*) FROM chunks').fetchone()[0]
n_null = db.execute('SELECT COUNT(*) FROM chunks WHERE vector_row_idx IS NULL').fetchone()[0]
print(f'chunk_vectors.npy: {chunk_vecs.shape}')
print(f'paper_vectors.npy: {paper_vecs.shape}')
print(f'chunks in DB: {n_chunks}, still unvectorised: {n_null}')
assert n_null == 0, 'some chunks never got a vector_row_idx'
assert chunk_vecs.shape[0] == n_chunks, 'vector count does not match chunk count'
print('OK')

## 7. Package for download

Bring these three files back into `data/`, replacing the local copies.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/embed_out', 'zip', '/content/data')
files.download('/content/embed_out.zip')